# Data Profiling and Data Cleaning Process

In [393]:
import pandas as pd
import numpy as np
df = pd.read_csv('data/raw/veltix_raw_data.csv')

## 1. Data Overview

In [394]:
df.shape

(8034, 10)

According to the data dictionary, the dataset should contain 7,800 rows — 234 extra rows observed here.

In [395]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8034 entries, 0 to 8033
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   sku_id             8034 non-null   str    
 1   product_category   8034 non-null   str    
 2   week_id            8034 non-null   str    
 3   sales_qty          7759 non-null   float64
 4   inventory_begin    7751 non-null   float64
 5   inventory_end      8034 non-null   int64  
 6   receipts           8034 non-null   int64  
 7   lead_time_days     8034 non-null   int64  
 8   unit_cost          8034 non-null   float64
 9   holding_cost_rate  8034 non-null   float64
dtypes: float64(4), int64(3), str(3)
memory usage: 627.8 KB


`sales_qty` and `inventory_begin` contain missing values.

In [396]:
df.sample(5)

,sku_id,product_category,week_id,sales_qty,inventory_begin,inventory_end,receipts,lead_time_days,unit_cost,holding_cost_rate
7109,SH-006,Smart Home,2024-W50,35.0,252.0,217,0,0,58.23,0.25
4162,AU-005,Audio,2023-W39,58.0,969.0,911,0,0,16.42,0.28
853,CP-007,Computer Peripherals,2024-W06,102.0,978.0,876,0,0,24.22,0.25
2847,AU-001,Audio,2024-W43,289.0,732.0,443,0,0,43.01,0.28
838,CP-010,Computer Peripherals,2025-W08,24.0,793.0,1028,259,32,41.81,0.25


In [397]:
df.describe()

,sales_qty,inventory_begin,inventory_end,receipts,lead_time_days,unit_cost,holding_cost_rate
count,7759.000000,7751.000000,8034.000000,8034.000000,8034.000000,8034.000000,8034.000000
mean,96.656915,1146.658496,1142.966517,93.372417,4.662061,45.121206,0.265985
std,125.048297,951.978088,951.384098,279.805016,10.959504,31.391823,0.020582
min,-351.000000,0.000000,0.000000,0.000000,0.000000,5.650000,0.250000
25%,31.000000,537.000000,536.250000,0.000000,0.000000,19.490000,0.250000
50%,61.000000,864.000000,861.000000,0.000000,0.000000,41.930000,0.250000
75%,126.000000,1370.000000,1358.000000,0.000000,0.000000,57.050000,0.280000
max,3530.000000,5145.000000,5145.000000,2303.000000,60.000000,131.410000,0.300000


According to the data dictionary, two columns contain values outside the expected range:
- `sales_qty`: observed -351 ~ 3530, but the valid range should be 0 ~ 600 (non-negative)
- `lead_time_days`: observed 0 ~ 60, but the maximum should not exceed 56

In [398]:
df.nunique()

sku_id                241
product_category       35
week_id               277
sales_qty             530
inventory_begin      2276
inventory_end        2323
receipts              756
lead_time_days         53
unit_cost              50
holding_cost_rate       3
dtype: int64

**Cardinality issues:**  
`sku_id`: 241 values (should be 50)  
`product_category`: 35 values (should be 5)  
`week_id`: 277 values (should be 156)  

---

## 2. Inspecting Key & Category Columns

In [399]:
df['product_category'].value_counts(dropna=False)


product_category
Smart Home               1524
Audio                    1522
Mobile Accessories       1510
Computer Peripherals     1509
Wearables                1499
WEARABLES                  27
mobile accessories         27
SMART HOME                 26
computer peripherals       25
Mob. Acc.                  23
wearables                  22
Smart Home Devices         22
PC Peripherals             21
audio                      20
AUDIO                      20
Audio Products             20
Wear.                      19
MOBILE ACCESSORIES         19
Comp. Periph.              19
COMPUTER PERIPHERALS       18
Wearable Devices           18
Aud.                       17
SmartHome                  16
smart home                 15
Mobile Acc.                11
 Wearables                  9
 Mobile Accessories         9
Wearables                   9
Computer Peripherals        8
 Smart Home                 6
Mobile Accessories          6
Smart Home                  5
Audio                  

In [400]:
df['sku_id'].value_counts(dropna=False)

sku_id
SH-003     161
MA-010     159
SH-002     158
AU-006     158
CP-006     158
          ... 
 CP-002      1
AU_010       1
WR_001       1
CP_006       1
MA-O05       1
Name: count, Length: 241, dtype: int64

In [401]:
df['week_id'].value_counts(dropna=False)

week_id
2025-W06    57
2025-W15    55
2024-W19    54
2024-W16    54
2023-W22    54
            ..
W26-2025     1
W09-2024     1
W35-2025     1
2023/W11     1
2023/W25     1
Name: count, Length: 277, dtype: int64

`product_category` issues: 
- casing (AUDIO/audio)
- abbreviations: Comp. Periph.
- leading/trailing spaces

`sku_id` issues: 
- underscore vs dash (AU_010)
- letter O vs zero (MA-O05)
- leading/trailing spaces

`week_id` issues: 
- reversed format (W26-2025)
- slash vs dash (2023/W11)
- may have leading/trailing spaces → clean all three together

### Clean leading and trailing spaces

In [402]:
# Count rows with leading/trailing spaces
cols = ['sku_id', 'product_category', 'week_id']
before = df[cols].apply(lambda x: x != x.str.strip()) 
print(f"Rows before cleaning: \n{before.sum()}")

Rows before cleaning: 
sku_id              49
product_category    65
week_id              0
dtype: int64


In [403]:
# Clean the rows in the three columns
df[cols] = df[cols].apply(lambda x: x.str.strip())
remaining = df[cols].apply(lambda x: x != x.str.strip())
print(f"Remaining rows with spaces: \n{remaining.sum()}")

Remaining rows with spaces: 
sku_id              0
product_category    0
week_id             0
dtype: int64


### Clean casing issues

In [404]:
# Count rows that are not in the title case format (audio, SMART HOME etc.)
before = df['product_category'] != df['product_category'].str.title()
print(f"Rows with casing issues: {before.sum()}")

Rows with casing issues: 256


In [405]:
# Normalize casing to title case
df['product_category'] = df['product_category'].str.title()
remaining = df['product_category'] != df['product_category'].str.title()
print(f"Remaining casing issues: {remaining.sum()} rows")

Remaining casing issues: 0 rows


In [406]:
# Count rows with abbreviated values (e.g., Comp. Periph.)
has_dot = df['product_category'].str.contains('.', regex= False)
print(f"Rows with abbreviated values: {has_dot.sum()}")

Rows with abbreviated values: 89


In [407]:
# List distinct abbreviated values
df.loc[has_dot, 'product_category'].value_counts()

product_category
Mob. Acc.        23
Wear.            19
Comp. Periph.    19
Aud.             17
Mobile Acc.      11
Name: count, dtype: int64

In [408]:
# Replace abbreviations with full names
mapping = {
    'Mob. Acc.': 'Mobile Accessories',
    'Wear.': 'Wearables',
    'Comp. Periph.': 'Computer Peripherals',
    'Aud.': 'Audio',
    'Mobile Acc.': 'Mobile Accessories'}

df['product_category'] = df['product_category'].replace(mapping)

In [409]:
# Varify result in `product_category`
df['product_category'].value_counts()

product_category
Mobile Accessories      1605
Audio                   1589
Wearables               1585
Computer Peripherals    1582
Smart Home              1576
Smart Home Devices        22
Pc Peripherals            21
Audio Products            20
Wearable Devices          18
Smarthome                 16
Name: count, dtype: int64

Discovered there are still 5 fomating issues in `product_category`

In [410]:
second_mapping = {
    'Smart Home Devices': 'Smart Home',
    'Pc Peripherals': 'Computer Peripherals',
    'Audio Products': 'Audio',
    'Wearable Devices': 'Wearables',
    'Smarthome': 'Smart Home'}

df['product_category'] = df['product_category'].replace(second_mapping)

In [411]:
df['product_category'].value_counts()

product_category
Smart Home              1614
Audio                   1609
Mobile Accessories      1605
Wearables               1603
Computer Peripherals    1603
Name: count, dtype: int64

Now `product_category` has a unified format. What's left are `sku_id` and `week_id`.

In [412]:
# Detect sku_id format issues: casing, underscore, letter O

not_upper = df['sku_id'] != df['sku_id'].str.upper()
has_underscore = df['sku_id'].str.contains('_')
has_letter_o = df['sku_id'].str.contains('O')

print(f"Not upper: {not_upper.sum()}")
print(f"Has underscore: {has_underscore.sum()}")
print(f"Rows with letter O: {has_letter_o.sum()}")

Not upper: 56
Has underscore: 90
Rows with letter O: 173


In [413]:
# Turn them into upper cases and replace '_' to '-'
df['sku_id'] = df['sku_id'].str.upper().str.replace('_', '-').str.replace('O', '0')


# Verify the result
not_upper = df['sku_id'] != df['sku_id'].str.upper()
has_underscore = df['sku_id'].str.contains('_')
has_letter_o = df['sku_id'].str.contains('O')

print(f"Not upper: {not_upper.sum()}")
print(f"Has underscore: {has_underscore.sum()}")
print(f"Rows with letter O: {has_letter_o.sum()}")



Not upper: 0
Has underscore: 0
Rows with letter O: 0


In [414]:
# Inspect `sku_id` values
df['sku_id'].value_counts()

sku_id
CP-006     165
MA-003     164
WR-006     164
SH-003     164
AU-008     163
          ... 
RNA-010      1
RNA-003      1
WR-00L       1
RNA-006      1
RNA-005      1
Name: count, Length: 64, dtype: int64

`sku_id` should contain 50 unique values but currently shows 64. Two remaining issues: 
- non-standard prefix (e.g., RNA-)
- letters in the suffix digits (e.g., WR-00L).

In [415]:
# Identify what `RNA` represents
df.loc[df['sku_id'].str.startswith('RNA')]

,sku_id,product_category,week_id,sales_qty,inventory_begin,inventory_end,receipts,lead_time_days,unit_cost,holding_cost_rate
74,RNA-007,Mobile Accessories,2023-W30,353.0,3086.0,2733,0,0,6.61,0.25
1994,RNA-010,Mobile Accessories,2024-W09,82.0,3203.0,3121,0,0,7.97,0.25
2071,RNA-003,Mobile Accessories,2024-W08,223.0,955.0,1659,927,16,5.65,0.25
2405,RNA-007,Mobile Accessories,2024/W16,516.0,2375.0,4135,2276,11,6.61,0.25
2655,RNA-001,Mobile Accessories,2025-W39,228.0,2647.0,2419,0,0,19.16,0.25
2973,RNA-001,Mobile Accessories,2025-W10,308.0,2815.0,2507,0,0,19.16,0.25
3830,RNA-009,Mobile Accessories,2024-W50,82.0,4591.0,4509,0,0,7.95,0.25
4718,RNA-009,Mobile Accessories,2025-W26,60.0,4837.0,4777,0,0,7.95,0.25
6007,RNA-006,Mobile Accessories,2025-W02,166.0,832.0,1662,996,19,19.49,0.25
7033,RNA-005,Mobile Accessories,2025-W43,332.0,2655.0,3474,1151,21,15.04,0.25


Every `RNA-` prefixed value falls under `Mobile Accessories`. Before mapping `RNA-` to `MA-`, I'll verify by comparing `unit_cost` — since cost is fixed per SKU, matching costs between `RNA-NNN` and `MA-NNN` would confirm they refer to the same product.

In [416]:
result = df.loc[df['sku_id'].str.startswith(('RNA-', 'MA-')), 
                ['sku_id', 'unit_cost']].drop_duplicates()
result['prefix'] = result['sku_id'].str[:-4]
result['suffix'] = result['sku_id'].str[-3:]
result.pivot(index='suffix', columns='prefix', values='unit_cost')

prefix,MA,RNA
suffix,,
001,19.16,19.16
002,13.18,NaN
003,5.65,5.65
004,17.72,NaN
005,15.04,15.04
006,19.49,19.49
007,6.61,6.61
008,8.17,NaN
009,7.95,7.95


Now that I've verified they refer to the same product, I'll replace `RNA-` with `MA-`.

In [417]:
mapping = {
    'RNA-001': 'MA-001',
    'RNA-003': 'MA-003',
    'RNA-005': 'MA-005',
    'RNA-006': 'MA-006',
    'RNA-007': 'MA-007',
    'RNA-009': 'MA-009',
    'RNA-010': 'MA-010',
}
df['sku_id'] = df['sku_id'].replace(mapping)

Next, I'll identify and clean values with letters in the suffix digits (e.g., `MA-00L`).

In [418]:
df[df['sku_id'].str.contains('L')]


,sku_id,product_category,week_id,sales_qty,inventory_begin,inventory_end,receipts,lead_time_days,unit_cost,holding_cost_rate
676,AU-0L0,Audio,2024-W01,38.0,1813.0,1775,0,0,57.05,0.28
826,AU-0L0,Audio,2023-W23,350.0,1964.0,1929,0,0,57.05,0.28
1062,WR-0L0,Wearables,2024-W26,8.0,657.0,649,0,0,38.20,0.30
1131,CP-00L,Computer Peripherals,2023-W07,125.0,1071.0,946,0,0,44.52,0.25
1227,AU-00L,Audio,2025-W08,88.0,1015.0,927,0,0,43.01,0.28
1335,AU-0L0,Audio,2024-W11,NaN,2010.0,1966,0,0,57.05,0.28
3544,MA-0L0,Mobile Accessories,2023-W34,64.0,3362.0,3298,0,0,7.97,0.25
4203,WR-00L,Wearables,2024-W49,166.0,833.0,667,0,0,87.21,0.30
4582,MA-00L,Mobile Accessories,2023-W19,209.0,3082.0,2873,0,0,19.16,0.25
5400,MA-0L0,Mobile Accessories,2024-W39,85.0,3568.0,3483,0,0,7.97,0.25


The same letter-in-suffix issue also appears in `Audio` and `Wearables`. I'll apply the same `unit_cost` comparison to determine which number they correspond to.

In [419]:
dirty = df.loc[df['sku_id'].str.contains('L'), 
               ['sku_id', 'unit_cost']].drop_duplicates()
valid = df.loc[~df['sku_id'].str.contains('L'), 
               ['sku_id', 'unit_cost']].drop_duplicates()

dirty['prefix'] = dirty['sku_id'].str[:2]
valid['prefix'] = valid['sku_id'].str[:2]
dirty.merge(valid, on=['prefix', 'unit_cost'], suffixes=('_dirty', '_clean'))

,sku_id_dirty,unit_cost,prefix,sku_id_clean
0,AU-0L0,57.05,AU,AU-010
1,WR-0L0,38.20,WR,WR-010
2,CP-00L,44.52,CP,CP-001
3,AU-00L,43.01,AU,AU-001
4,MA-0L0,7.97,MA,MA-010
5,WR-00L,87.21,WR,WR-001
6,MA-00L,19.16,MA,MA-001


The `L` characters in the suffix are visual confusion errors,every dirty value cross-validates to a `1` in the same position. Verified by joining each dirty SKU against the clean set on `(prefix, unit_cost)`, since `unit_cost` is fixed per SKU.

In [420]:
# Replace L characters with 1, then verify the unique values of `sku_id`
df['sku_id'] = df['sku_id'].str.replace('L', '1')
print(np.sort(df['sku_id'].unique()))
print(f"Count: {df['sku_id'].nunique()}")

['AU-001' 'AU-002' 'AU-003' 'AU-004' 'AU-005' 'AU-006' 'AU-007' 'AU-008'
 'AU-009' 'AU-010' 'CP-001' 'CP-002' 'CP-003' 'CP-004' 'CP-005' 'CP-006'
 'CP-007' 'CP-008' 'CP-009' 'CP-010' 'MA-001' 'MA-002' 'MA-003' 'MA-004'
 'MA-005' 'MA-006' 'MA-007' 'MA-008' 'MA-009' 'MA-010' 'SH-001' 'SH-002'
 'SH-003' 'SH-004' 'SH-005' 'SH-006' 'SH-007' 'SH-008' 'SH-009' 'SH-010'
 'WR-001' 'WR-002' 'WR-003' 'WR-004' 'WR-005' 'WR-006' 'WR-007' 'WR-008'
 'WR-009' 'WR-010']
Count: 50


All 5 category prefixes have 10 unique suffixes each, totaling 50 unique SKUs as defined in the data dictionary. Next up: `week_id`.

In [421]:
# Identify values that contain '/' instead of '-'
df.loc[df['week_id'].str.contains('/'), 'week_id'].unique()

<StringArray>
['2024/W43', '2024/W27', '2025/W04', '2025/W11', '2024/W23', '2024/W48',
 '2024/W40', '2024/W31', '2023/W41', '2024/W32', '2025/W49', '2024/W03',
 '2023/W20', '2024/W16', '2025/W35', '2024/W42', '2024/W21', '2023/W13',
 '2023/W05', '2024/W08', '2024/W47', '2024/W20', '2025/W39', '2023/W44',
 '2023/W30', '2023/W15', '2023/W42', '2023/W37', '2023/W43', '2025/W44',
 '2023/W02', '2025/W10', '2025/W33', '2025/W25', '2024/W49', '2025/W37',
 '2025/W16', '2023/W12', '2025/W05', '2024/W11', '2023/W48', '2025/W50',
 '2024/W29', '2025/W30', '2024/W18', '2024/W17', '2025/W09', '2024/W45',
 '2025/W51', '2023/W51', '2024/W37', '2023/W03', '2024/W35', '2023/W11',
 '2023/W25']
Length: 55, dtype: str

In [422]:
# Repale '/' to '-' separator
df['week_id'] = df['week_id'].str.replace('/', '-')
df.loc[df['week_id'].str.contains('/'), 'week_id'].unique()

<StringArray>
[]
Length: 0, dtype: str

In [423]:
# Identify reversed format values in `week_id`
mask = df['week_id'].str.startswith('W')
df.loc[mask, 'week_id'].unique()

<StringArray>
['W45-2025', 'W01-2024', 'W42-2025', 'W43-2023', 'W38-2023', 'W38-2025',
 'W33-2023', 'W04-2025', 'W48-2024', 'W10-2023', 'W21-2024', 'W41-2023',
 'W19-2025', 'W22-2024', 'W39-2024', 'W48-2025', 'W24-2023', 'W47-2024',
 'W26-2023', 'W31-2025', 'W32-2024', 'W18-2024', 'W30-2024', 'W09-2025',
 'W16-2025', 'W13-2025', 'W20-2025', 'W50-2025', 'W45-2023', 'W12-2024',
 'W25-2025', 'W51-2025', 'W46-2023', 'W25-2024', 'W35-2024', 'W36-2024',
 'W46-2025', 'W27-2023', 'W28-2025', 'W28-2024', 'W30-2025', 'W44-2025',
 'W21-2025', 'W35-2023', 'W51-2024', 'W37-2025', 'W47-2023', 'W17-2024',
 'W26-2024', 'W02-2025', 'W23-2025', 'W17-2025', 'W02-2024', 'W06-2023',
 'W46-2024', 'W22-2025', 'W39-2025', 'W26-2025', 'W09-2024', 'W35-2025']
Length: 60, dtype: str

In [424]:
# Swap year and week so the format becomes YYYY-Wnn
mask = df['week_id'].str.startswith('W')
df.loc[mask, 'week_id'] = df.loc[mask, 'week_id'].str.split('-').apply(lambda x: f"{x[1]}-{x[0]}")
df['week_id'].value_counts()

week_id
2025-W06    57
2024-W16    55
2025-W13    55
2025-W15    55
2023-W45    54
            ..
2024-W3      1
2024-W8      1
2024-W4      1
2023-W2      1
2024-W9      1
Name: count, Length: 162, dtype: int64

After cleaning, `week_id` has 162 unique values instead of the expected 156. Likely cause: single-digit weeks missing a leading zero (e.g., W5 should be W05), inflating the unique count.

In [425]:
# Identify values with missing leading zero of sigle-digit weeks
mask = df['week_id'].str.match(r'.*-W\d$')
df.loc[mask, 'week_id'].unique()

<StringArray>
['2024-W3', '2024-W8', '2024-W7', '2024-W4', '2023-W2', '2024-W9']
Length: 6, dtype: str

In [426]:
mapping = {
    '2024-W3': '2024-W03',
    '2024-W8': '2024-W08',
    '2024-W7': '2024-W07',
    '2024-W4': '2024-W04',
    '2023-W2': '2023-W02',
    '2024-W9': '2024-W09',
}
df['week_id'] = df['week_id'].replace(mapping)
df['week_id'].nunique()

156

All key and categorical columns are now in the correct format, with unique value counts matching the data dictionary:  
- sku_id: 50
- product_category: 5
- week_id: 156  

Next stage: missing values, duplicate rows, and invalid-value checks (e.g., negative quantities, out-of-range lead times).

## 3. Duplicate rows, Missing values, and Invalid-value checks

### Duplicates
---

In [427]:
# Inspect duplicate rows by sorting on key columns so paired duplicates appear adjacent.
df[df.duplicated(keep=False)].sort_values(['sku_id', 'week_id'])

,sku_id,product_category,week_id,sales_qty,inventory_begin,inventory_end,receipts,lead_time_days,unit_cost,holding_cost_rate
4424,AU-001,Audio,2024-W16,81.0,1035.0,954,0,0,43.01,0.28
6394,AU-001,Audio,2024-W16,81.0,1035.0,954,0,0,43.01,0.28
1338,AU-001,Audio,2025-W49,194.0,1131.0,937,0,0,43.01,0.28
5771,AU-001,Audio,2025-W49,194.0,1131.0,937,0,0,43.01,0.28
778,AU-002,Audio,2023-W26,116.0,1314.0,1198,0,0,25.58,0.28
...,...,...,...,...,...,...,...,...,...,...
4974,WR-010,Wearables,2024-W32,13.0,685.0,672,0,0,38.20,0.30
7611,WR-010,Wearables,2025-W32,NaN,706.0,695,0,0,38.20,0.30
7767,WR-010,Wearables,2025-W32,NaN,706.0,695,0,0,38.20,0.30
3766,WR-010,Wearables,2025-W46,37.0,600.0,563,0,0,38.20,0.30


In [428]:
# Number of duplicates rows
df[df.duplicated()].sort_values(['sku_id', 'week_id']).shape

(234, 10)

Visual inspection isn't enough, some could be key collisions with different values. If all-column duplicates equal key-only duplicates, every duplicate is a complete copy.

In [429]:
all_dups = df.duplicated().sum()                                  
key_dups = df.duplicated(subset=['sku_id', 'week_id']).sum()      
print(f"Completely identical: {all_dups}, key columns same: {key_dups}")

Completely identical: 234, key columns same: 234


Both counts match (234), confirming all duplicates are exact copies. Safe to drop with drop_duplicates(), expecting row count to fall from 8034 to 7800.

In [430]:
df = df.drop_duplicates()
df.shape

(7800, 10)

After removing duplicates, the row count matches the expected total from the data dictionary (50 SKUs × 156 weeks = 7,800 records).

### Missing values
---

In [431]:
df.isna().sum()

sku_id                 0
product_category       0
week_id                0
sales_qty            266
inventory_begin      275
inventory_end          0
receipts               0
lead_time_days         0
unit_cost              0
holding_cost_rate      0
dtype: int64

Only two columns have missing values. Before deciding whether to drop them, I'll check if the missing values can be recovered from the other three columns using the inventory equation: ```inventory_end = inventory_begin + receipts - sales_qty```.

In [432]:
# Check whether any rows have both sales_qty and inventory_begin missing
both_missing = df['sales_qty'].isna() & df['inventory_begin'].isna()
print(f"Both missing: {both_missing.sum()}")


Both missing: 0


No overlap, every missing sales_qty has a non-null inventory_begin (and vice versa). That means each missing value could in theory be recovered from the other three columns via `inventory_end = inventory_begin + receipts - sales_qty`.

But this only works if the equation actually holds in the data. If it doesn't, I'd be filling NaNs with values that silently inherit business logic errors. Better to validate the equation first, then come back to missing values.

### Invalid logic check
---
The inventory equation `inventory_end = inventory_begin + receipts - sales_qty` should hold for every row. Validate it on rows where all four columns are present, these serve as ground truth before using the equation to recover missing values.

In [435]:
df[df['sales_qty'].isna()]

,sku_id,product_category,week_id,sales_qty,inventory_begin,inventory_end,receipts,lead_time_days,unit_cost,holding_cost_rate
27,CP-009,Computer Peripherals,2024-W02,NaN,1391.0,1358,0,0,49.81,0.25
57,AU-010,Audio,2023-W24,NaN,1929.0,1883,0,0,57.05,0.28
66,WR-008,Wearables,2024-W39,NaN,739.0,725,0,0,121.16,0.30
90,AU-005,Audio,2025-W01,NaN,658.0,605,0,0,16.42,0.28
125,SH-009,Smart Home,2023-W20,NaN,306.0,300,0,0,28.70,0.25
...,...,...,...,...,...,...,...,...,...,...
7962,AU-004,Audio,2023-W14,NaN,658.0,598,0,0,53.31,0.28
7973,WR-005,Wearables,2025-W43,NaN,940.0,728,0,0,131.41,0.30
7988,AU-008,Audio,2024-W44,NaN,926.0,893,0,0,41.93,0.28
8008,SH-009,Smart Home,2024-W42,NaN,342.0,334,0,0,28.70,0.25
